# VideoJev-4B をベンチマーク 722 問の時刻の課題で評価する（Colab A100）

VideoJev（南京大、arXiv 2610.18615、Apache-2.0）は、文章を生成する代わりに「区間ごとに該当する確率」を付ける頭を Qwen3.5-4B に足し、問い + 正解区間 19 万件で教師あり学習したモデル。
同じ 722 問で、GRPO 後の Qwen3-VL-4B（62.2%）と並べる。採点は同じ（答え = 区間の中点が正解区間 ± 許容幅に入るか。許容幅は 32 フレーム相当）。

前提: Drive の `MyDrive/movie_llm/` に `videos_2fps_s360.zip`（手元で `scripts/jvqa_reencode.py` で小さくした動画 389 本）がある。VideoJev は動画ファイルを直接読む（フレームキャッシュは使えない）。

注意: transformers は VideoJev が要求する 5.14.1 を入れる（Qwen3.5 のクラスが要る）。flash-attn は入れずに sdpa で動かす。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers==5.14.1" "accelerate>=1.10" "einops>=0.8" "scipy>=1.12" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "safetensors>=0.4"
!pip install -q "flash-linear-attention==0.4.2" "fla-core==0.4.2" || echo 'fla なしで続行（遅いが動く）'
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps
import transformers, torch; print(transformers.__version__, torch.__version__)

## 2. Drive と動画

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os
!mkdir -p /content/videos_small
!unzip -q -o {DRIVE}/videos_2fps_s360.zip -d /content/videos_small
print('動画', len(os.listdir('/content/videos_small')), '本')

## 3. 動作確認（3 本だけ）

出力の「例: 問い → [start, end]」が秒の区間になっていれば OK。`None` ばかりなら threshold を下げる（`--threshold 0.1`）。

In [ ]:
%cd /content/movie_llm
!rm -rf /content/vj_smoke
!python scripts/videojev_predict.py --name videojev-4b --video-dir /content/videos_small --out-dir /content/vj_smoke --fps 2 --attn sdpa --limit 3
!head -c 1500 /content/vj_smoke/videojev-4b/predictions.jsonl

## 4. 全 722 問（約 1〜2 時間。途中で切れたら同じセルを再実行すると続きから）

2 回目（名前 videojev-4b-p）は候補ごとの確率も保存する。信頼度で自動 / 候補提示 / 該当なしに振り分けるしきい値を決めるため。

In [ ]:
%cd /content/movie_llm
!python scripts/videojev_predict.py --name videojev-4b-p --video-dir /content/videos_small --out-dir {DRIVE}/results --fps 2 --attn sdpa
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/videojev-4b-p/predictions.jsonl --name videojev-4b-p --out-root {DRIVE}/results

終わったら表（全体・short・medium）を貼り、`predictions.jsonl` を手元の `results/videojev-4b/` に置く。区間そのもの（interval）も記録しているので、手元で「区間と正解区間の重なり」でも採点する。